This is a notebook for extracting embeddings (and tokenizer) from minishlab/potion-multilingual-128M. We supply this instead of embeddings, because they exceed file size permitted by github.

In [2]:
#get an instance of potion model
from model2vec import StaticModel
staticModel = StaticModel.from_pretrained("minishlab/potion-multilingual-128M")

from sentence_transformers import SentenceTransformer
sentTfModel = SentenceTransformer("minishlab/potion-multilingual-128M")
sentTfModel.to("cpu")

#load tokenizer
from transformers import XLMRobertaTokenizerFast
tokenizer = XLMRobertaTokenizerFast.from_pretrained("minishlab/potion-multilingual-128M")

In [ ]:
#save a copy of original tokenizer and print added tokens

#XXX: do not uncomment this, or it will overwrite hand made modifications
# tokenizer.save_pretrained("./data/tokenizer-fix")

display(tokenizer.added_tokens_decoder)

#we modify the saved files manually as per suggestion in https://github.com/huggingface/transformers/issues/27974 to fix indices of <pad> and <unk> tokens and remove [pad] and [unk] tokens

{0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 500353: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 500354: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 500355: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 500356: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 500357: AddedToken("<mask>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True)}

In [ ]:
#check if fixed tokenizer operates as expected
import os.path
tokenizerFixed = XLMRobertaTokenizerFast.from_pretrained("./data/tokenizer-fix")
display(tokenizerFixed.added_tokens_decoder)
display(tokenizerFixed.added_tokens_encoder)

enc = tokenizerFixed("some text", max_length=10, padding='max_length')
display(enc)
print(tokenizerFixed.convert_ids_to_tokens(enc.input_ids))

{0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 1: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 500353: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 500354: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 500355: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=False, special=True)}

{'<pad>': 0, '<unk>': 1, '<s>': 500353, '</s>': 500354, '<mask>': 500355}

{'input_ids': [3058, 7984, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 0, 0, 0, 0, 0, 0, 0, 0]}

['▁some', '▁text', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


In [19]:
#extract potion embeddings as torch embeddings
import torch
import torch.nn as tnn

embBag = sentTfModel.get_submodule("0.embedding")
padding_idx = tokenizerFixed.convert_tokens_to_ids(["<pad>"])[0]
print(padding_idx)
emb = tnn.Embedding.from_pretrained(embBag.weight, padding_idx=padding_idx)

with open("./data/embeddings/save.torch_save", "wb") as f:
    torch.save(emb, f)

0


In [30]:
#check if fixed-tokenizer + extracted embeddings work the same as original embedder

text = "some text with some words"

ids = tokenizerFixed(text, padding='do_not_pad').input_ids
ids = torch.tensor(ids)
encsExtr = emb.forward(ids)

encsOrig = staticModel.encode_as_sequence(text)
encsOrig = torch.tensor(encsOrig)

print((encsExtr == encsOrig).all())

tensor(True)


In [37]:
#check what happens if we start passing special tokens into text
tokenizerFixed.split_special_tokens = True
enc = tokenizerFixed("some <unk> some <pad> some <s>", padding='do_not_pad')
display(enc)
print(tokenizerFixed.convert_ids_to_tokens(enc.input_ids))

{'input_ids': [3058, 4424, 49, 90, 975, 3058, 4424, 7920, 975, 3058, 4424, 89, 975], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

['▁some', '▁<', '▁un', 'k', '▁>', '▁some', '▁<', '▁pad', '▁>', '▁some', '▁<', '▁s', '▁>']
